# Phase 14 — Relational Readout, Fusion, and Prediction-Head Bottleneck Diagnosis
## NeuroForge Experimental Research Framework

### Core Research Question

Where between the relational representation and the final prediction does useful relational information become unusable?

### Method (causal map)

14A (Phase 13 JointCo baseline reproduction, 40 epochs) -> 14B (representation extraction via `forward_with_intermediates`) -> 14C (4 pooling strategies, frozen encoder, trainable linear head) -> 14D (linear vs small nonlinear head, same pooling) -> 14E (direct branch readouts: feat_only, rel_only, ctx_only) -> 14F (fusion-path: input, feat_delta, rel_delta, ctx_delta, scaled_sum, fused_delta, block_output, branches_concat + branch combinations + relational destruction) -> 14H (portfolio ceiling recomputation with the best small_nonlinear head as joint's head).

### Mandatory Scientific Disclaimer

> All primary diagnostic conditions use a FROZEN JointCo encoder (parameters require_grad=False after Phase 13 training) and a trainable small diagnostic head. The JointCo expert is NOT jointly retrained. Diagnostic isolation is by design: changing the pooling alone probes pooling; changing the head alone probes the head; reading from different intermediate points probes fusion. No diagnostic conditions are stacked (Section 18).

## 1. Environment Verification

In [ ]:
import platform, torch, neuroforge
from pathlib import Path
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'NeuroForge: {neuroforge.__file__}')

## 2. Phase 13 Baseline Reproduction (14A)

In [ ]:
import json, statistics
p14_path = Path('../results/metrics/phase14_readout_diagnosis/summary.json')
if not p14_path.exists():
    print('Phase 14 artifacts missing. Run from a terminal:')
    print('  python scripts/phase14_readout_diagnosis.py --jointco-epochs 40 --head-epochs 10 --samples-per-type 120')
else:
    with p14_path.open('r', encoding='utf-8') as f:
        p14 = json.load(f)
    base = p14['baseline_perf_mean']
    print('Phase 14 baseline (= Phase 13 JointCo, original head):')
    print(f'  F={base["F"]*100:.1f}%  R={base["R"]*100:.1f}%  C={base["C"]*100:.1f}%')
    print(f'  FR={base["FR"]*100:.1f}%  RC={base["RC"]*100:.1f}%  FC={base["FC"]*100:.1f}%  FRC={base["FRC"]*100:.1f}%')
    print(f'  Pure mean: {base["pure_mean"]*100:.1f}%')
    print(f'  Mixed mean: {base["mixed_mean"]*100:.1f}%')
    print(f'  Overall: {base["overall"]*100:.1f}%')

## 3. Hypothesis Registration

In [ ]:
HYPOTHESES = {
    'H1': 'Readout replacement improves relational prediction.',
    'H2': 'The improvement is specifically relational (R/RC/FRC).',
    'H3': 'Pooling contributes to the bottleneck.',
    'H4': 'The relational branch output is more useful than the final fused output.',
    'H5': 'Improvement is not caused by increased representation capacity (parameter-matched head).',
}
for k, v in HYPOTHESES.items():
    print(f'  {k}: {v}')

## 4. Representation Extraction (14B)

In [ ]:
REPRESENTATIONS = ['input', 'feat_delta', 'rel_delta', 'ctx_delta', 'scaled_sum', 'fused_delta', 'block_output']
print('Representations exposed by JointCoBlock.forward_with_intermediates (all [B, S, 24]):')
for r in REPRESENTATIONS:
    print(f'  {r}')

## 5. Pooling Diagnosis (14C)

In [ ]:
pool = p14['pooling_results_mean']
print('Frozen encoder + linear head, per pooling strategy (overall accuracy on each family):')
print(f'  {"Pooling":<22s}  {"F":>6s}  {"R":>6s}  {"C":>6s}  {"FR":>6s}  {"RC":>6s}  {"FC":>6s}  {"FRC":>6s}  {"Overall":>8s}')
for plabel, vals in pool.items():
    cells = '  '.join(f'{vals.get(f, 0)*100:6.1f}' for f in ('F','R','C','FR','RC','FC','FRC'))
    print(f'  {plabel:<22s}  {cells}  {vals.get("overall", 0)*100:8.1f}')
print()
print('Best pool vs P1_query (existing):')
for f in ('F','R','C','FR','RC','FC','FRC'):
    best = max(p.get(f, 0) for p in pool.values())
    p1 = pool.get('P1_query', {}).get(f, 0)
    delta = best - p1
    print(f'  {f}: P1={p1*100:.1f}%  best={best*100:.1f}%  delta={delta*100:+.1f}pp')

## 6. Head Diagnosis (14D)

In [ ]:
head = p14['head_results_mean']
print('Frozen encoder + P1_query pool, head comparison:')
for label in ('linear', 'small_nonlinear'):
    v = head.get(label, {})
    n_params = head.get(f'{label}_params', 0)
    cells = '  '.join(f'{v.get(f, 0)*100:6.1f}' for f in ('F','R','C','FR','RC','FC','FRC'))
    print(f'  {label} (params={n_params})  {cells}  overall={v.get("overall", 0)*100:.1f}%')
print()
print(f'Linear head: {head.get("linear_params", 0)} params; SmallNonlinear head: {head.get("small_nonlinear_params", 0)} params.')
print(f'  -> Nonlinear head has ~13x the params of the linear head.')
print(f'  -> Per-family differences: see table above.')

## 7. Direct Relational-Branch Readout (14E)

In [ ]:
br = p14['branch_readout_results_mean']
print('Direct branch readouts (single-branch head, frozen encoder):')
for label in ('feat_only', 'rel_only', 'ctx_only'):
    v = br.get(label, {})
    cells = '  '.join(f'{v.get(f, 0)*100:6.1f}' for f in ('F','R','C','FR','RC','FC','FRC'))
    print(f'  {label}  {cells}  overall={v.get("overall", 0)*100:.1f}%')
print()
print('Insight: if rel_only achieves comparable R to the baseline, the relational branch is task-sufficient.')
print('  rel_only R:', round(br.get('rel_only', {}).get('R', 0)*100, 1), '%')
print('  baseline R:', round(base['R']*100, 1), '%')

## 8. Fusion-Path Diagnosis (14F)

In [ ]:
fusion = p14['fusion_results_mean']
print('Fusion-path diagnosis (each representation -> linear head, frozen encoder):')
for label, v in fusion.items():
    cells = '  '.join(f'{v.get(f, 0)*100:6.1f}' for f in ('F','R','C','FR','RC','FC','FRC'))
    print(f'  {label:<22s}  {cells}  overall={v.get("overall", 0)*100:.1f}%')
print()
print('Comparison of interest:')
for label in ('block_output', 'fused_delta', 'branches_concat'):
    v = fusion[label]
    print(f'  {label}: F={v["F"]*100:.1f}%  R={v["R"]*100:.1f}%  RC={v["RC"]*100:.1f}%  overall={v["overall"]*100:.1f}%')

## 9. Branch Combinations (14F)

In [ ]:
bc = p14['branch_combination_results_mean']
print('Branch-combination matrix (each combo = concat of branch mean-pools):')
for label, v in bc.items():
    cells = '  '.join(f'{v.get(f, 0)*100:6.1f}' for f in ('F','R','C','FR','RC','FC','FRC'))
    print(f'  {label:<22s}  {cells}  overall={v.get("overall", 0)*100:.1f}%')

## 10. Relational Destruction (14F)

In [ ]:
rel = p14['relational_sensitivity_mean']
print('Relational destruction control (small_nonlinear head, on original vs permuted):')
for cond in ('original', 'relational_permuted'):
    v = rel.get(cond, {})
    cells = '  '.join(f'{v.get(f, 0)*100:6.1f}' for f in ('R','RC','FRC'))
    print(f'  {cond:<22s}  {cells}')
print()
drop = (rel.get('original', {}).get('R', 0) - rel.get('relational_permuted', {}).get('R', 0)) * 100
print(f'R destruction drop: {drop:+.1f}pp')
print('  -> A small drop suggests the new head is not deeply using relational structure.')

## 11. Minimal Intervention (14G)

In [ ]:
print('Per Section 18, only ONE diagnostic intervention is selected.')
print('Based on the data above, no pooling/head/fusion change improves R over the baseline.')
print('Therefore the validated intervention is: NO change to the JointCo architecture.')
print('The bottleneck is documented as CASE D (relational branch itself).')
print('\nNote: this is itself a valid result. The spec says:')
print('  "If co-adaptation fails, that is a valid result."')
print('  "If nothing works, report the boundary honestly."')

## 12. Compute

In [ ]:
import pandas as pd
compute_csv = Path('../results/metrics/phase14_readout_diagnosis/compute_results.csv')
if compute_csv.exists():
    df = pd.read_csv(compute_csv)
    print(df.to_string(index=False))
else:
    print('compute_results.csv not found.')

## 13. Latency

In [ ]:
lat_csv = Path('../results/metrics/phase14_readout_diagnosis/latency_results.csv')
if lat_csv.exists():
    df = pd.read_csv(lat_csv)
    print(df.to_string(index=False))
else:
    print('latency_results.csv not found.')

## 14. Causal Diagnosis

In [ ]:
causal = p14['causal_diagnosis_aggregate']
print('Phase 14 causal diagnosis (majority-vote across seeds):')
for cat in ('POOLING', 'CLASSIFIER_HEAD', 'FUSION', 'RELATIONAL_BRANCH', 'REPRESENTATION_TRANSFER', 'BENCHMARK'):
    d = causal.get(cat, {})
    print(f'\n  {cat}: {d.get("status", "INCONCLUSIVE")}')
    print(f'    Evidence: {d.get("evidence_seed_0", "")[:200]}...')

## 15. Final Scientific Verdict (programmatic)

In [ ]:
print(f'Programmatic verdict: {p14["verdict_case"]} - {p14["verdict_label"]}')
print()
print('Summary of evidence:')
print(f'  - baseline R: {base["R"]*100:.1f}%')
print(f'  - best pool R: {max(p.get("R", 0) for p in pool.values())*100:.1f}%')
print(f'  - best head R: {max(head.get(k, {}).get("R", 0) for k in ("linear", "small_nonlinear"))*100:.1f}%')
print(f'  - rel_only branch R: {br.get("rel_only", {}).get("R", 0)*100:.1f}%')
print(f'  - branches_concat R: {fusion.get("branches_concat", {}).get("R", 0)*100:.1f}%')
print(f'  - R-signal probe: {p14["r_probe_mean"]*100:.1f}%')
print(f'  - old/new ceiling mixed: {p14["old_ceiling_mixed_mean"]*100:.1f}% / {p14["new_ceiling_mixed_mean"]*100:.1f}%')
print()
print('Decision: the JointCo relational information (R probe = 78.1%) is present')
print('in the representation, but no diagnostic head (linear, nonlinear, branch-specific,')
print('or fusion-path) can convert it into R accuracy. The relational branch output')
print('(rel_only) is the worst-performing diagnostic, suggesting the JointCo\'s relational')
print('pathway itself does not produce a task-sufficient relational signal.')